In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
import os
import json

In [27]:
# ==========================================
# 1. DATABASE CONNECTION & DATA FETCHING
# ==========================================
def get_db_connection():
    # SQLAlchemy format for Windows Auth: postgresql://@host:port/dbname
    # No username or password needed
    engine = create_engine('postgresql://@priorpsrv03:5432/gis')
    return engine

def fetch_data(schema, layer, columns):
    engine = get_db_connection()
    
    # NEW: Wrap every column name in double quotes so SQL doesn't panic over numbers
    quoted_columns = [f'"{col}"' for col in columns]
    col_string = ", ".join(quoted_columns)
    
    # Query ONLY the columns we need, completely ignoring geometry
    query = f"SELECT {col_string} FROM {schema}.{layer}"
    print(f"Executing query: SELECT [Needed Columns] FROM {schema}.{layer}")
    
    with engine.connect() as conn:
        df = pd.read_sql(text(query), conn)
    return df

In [25]:
# ==========================================
# 2. CONFIGURATIONS
# ==========================================
ethnicity_config = {
    # Broad Groups
    "asian_perc": {"source": "asian_count", "denominator": "total_ethnicity_pop", "method": "percentage"},
    "black_perc": {"source": "black_count", "denominator": "total_ethnicity_pop", "method": "percentage"},
    "mixed_or_multiple_perc": {"source": "mixed_or_multiple_count", "denominator": "total_ethnicity_pop", "method": "percentage"},
    "other_ethnic_perc": {"source": "other_ethnic_count", "denominator": "total_ethnicity_pop", "method": "percentage"},
    "white_perc": {"source": "white_count", "denominator": "total_ethnicity_pop", "method": "percentage"},
    
    # Detailed Groups - Asian
    "asian_bangladeshi_perc": {"source": "asian_bangladeshi_count", "denominator": "total_ethnicity_pop", "method": "percentage"},
    "asian_chinese_perc": {"source": "asian_chinese_count", "denominator": "total_ethnicity_pop", "method": "percentage"},
    "asian_indian_perc": {"source": "asian_indian_count", "denominator": "total_ethnicity_pop", "method": "percentage"},
    "asian_other_perc": {"source": "asian_other_count", "denominator": "total_ethnicity_pop", "method": "percentage"},
    "asian_pakistani_perc": {"source": "asian_pakistani_count", "denominator": "total_ethnicity_pop", "method": "percentage"},
    
    # Detailed Groups - Black
    "black_african_perc": {"source": "black_african_count", "denominator": "total_ethnicity_pop", "method": "percentage"},
    "black_caribbean_perc": {"source": "black_caribbean_count", "denominator": "total_ethnicity_pop", "method": "percentage"},
    "black_other_perc": {"source": "black_other_count", "denominator": "total_ethnicity_pop", "method": "percentage"},
    
    # Detailed Groups - Mixed
    "mixed_or_multiple_other_perc": {"source": "mixed_or_multiple_other_count", "denominator": "total_ethnicity_pop", "method": "percentage"},
    "mixed_or_multiple_white_and_asian_perc": {"source": "mixed_or_multiple_white_and_asian_count", "denominator": "total_ethnicity_pop", "method": "percentage"},
    "mixed_or_multiple_white_and_black_african_perc": {"source": "mixed_or_multiple_white_and_black_african_count", "denominator": "total_ethnicity_pop", "method": "percentage"},
    "mixed_or_multiple_white_and_black_caribbean_perc": {"source": "mixed_or_multiple_white_and_black_caribbean_count", "denominator": "total_ethnicity_pop", "method": "percentage"},
    
    # Detailed Groups - Other
    "other_other_perc": {"source": "other_other_count", "denominator": "total_ethnicity_pop", "method": "percentage"},
    "other_arab_perc": {"source": "other_arab_count", "denominator": "total_ethnicity_pop", "method": "percentage"},
    
    # Detailed Groups - White
    "white_british_perc": {"source": "white_british_count", "denominator": "total_ethnicity_pop", "method": "percentage"},
    "white_gypsy_or_irish_perc": {"source": "white_gypsy_or_irish_count", "denominator": "total_ethnicity_pop", "method": "percentage"},
    "white_irish_perc": {"source": "white_irish_count", "denominator": "total_ethnicity_pop", "method": "percentage"},
    "white_other_perc": {"source": "white_other_count", "denominator": "total_ethnicity_pop", "method": "percentage"},
    "white_roma_perc": {"source": "white_roma_count", "denominator": "total_ethnicity_pop", "method": "percentage"}
}

# Religion
religion_config = {
    "buddhist_perc": {"source": "buddhist_count", "denominator": "total_religion_pop", "method": "percentage"},
    "christian_perc": {"source": "christian_count", "denominator": "total_religion_pop", "method": "percentage"},
    "hindu_perc": {"source": "hindu_count", "denominator": "total_religion_pop", "method": "percentage"},
    "jewish_perc": {"source": "jewish_count", "denominator": "total_religion_pop", "method": "percentage"},
    "muslim_perc": {"source": "muslim_count", "denominator": "total_religion_pop", "method": "percentage"},
    "no_religion_perc": {"source": "no_religion_count", "denominator": "total_religion_pop", "method": "percentage"},
    "not_answered_perc": {"source": "not_answered_count", "denominator": "total_religion_pop", "method": "percentage"},
    "other_religion_perc": {"source": "other_religion_count", "denominator": "total_religion_pop", "method": "percentage"},
    "sikh_perc": {"source": "sikh_count", "denominator": "total_religion_pop", "method": "percentage"}
}

#Education
education_config = {
    # 1. Total Population
    "no_qualifications_perc": {"source": "no_qualifications_count", "denominator": "total_edu_pop_count", "method": "percentage"},
    "level_1_and_entry_level_qualifications_perc": {"source": "level_1_and_entry_level_qualifications_count", "denominator": "total_edu_pop_count", "method": "percentage"},
    "level_2_qualifications_perc": {"source": "level_2_qualifications_count", "denominator": "total_edu_pop_count", "method": "percentage"},
    "apprenticeship_perc": {"source": "apprenticeship_count", "denominator": "total_edu_pop_count", "method": "percentage"},
    "level_3_qualifications_perc": {"source": "level_3_qualifications_count", "denominator": "total_edu_pop_count", "method": "percentage"},
    "level_4_qualifications_or_above_perc": {"source": "level_4_qualifications_or_above_count", "denominator": "total_edu_pop_count", "method": "percentage"},
    "other_qualifications_perc": {"source": "other_qualifications_count", "denominator": "total_edu_pop_count", "method": "percentage"},

    # 2. Female Population
    "no_qualifications_female_perc": {"source": "no_qualifications_female_count", "denominator": "total_female_edu_pop_count", "method": "percentage"},
    "level_1_and_entry_level_qualifications_female_perc": {"source": "level_1_and_entry_level_qualifications_female_count", "denominator": "total_female_edu_pop_count", "method": "percentage"},
    "level_2_qualifications_female_perc": {"source": "level_2_qualifications_female_count", "denominator": "total_female_edu_pop_count", "method": "percentage"},
    "apprenticeship_female_perc": {"source": "apprenticeship_female_count", "denominator": "total_female_edu_pop_count", "method": "percentage"},
    "level_3_qualifications_female_perc": {"source": "level_3_qualifications_female_count", "denominator": "total_female_edu_pop_count", "method": "percentage"},
    "level_4_qualifications_or_above_female_perc": {"source": "level_4_qualifications_or_above_female_count", "denominator": "total_female_edu_pop_count", "method": "percentage"},
    "other_qualifications_female_perc": {"source": "other_qualifications_female_count", "denominator": "total_female_edu_pop_count", "method": "percentage"},

    # 3. Male Population
    "no_qualifications_male_perc": {"source": "no_qualifications_male_count", "denominator": "total_male_edu_pop_count", "method": "percentage"},
    "level_1_and_entry_level_qualifications_male_perc": {"source": "level_1_and_entry_level_qualifications_male_count", "denominator": "total_male_edu_pop_count", "method": "percentage"},
    "level_2_qualifications_male_perc": {"source": "level_2_qualifications_male_count", "denominator": "total_male_edu_pop_count", "method": "percentage"},
    "apprenticeship_male_perc": {"source": "apprenticeship_male_count", "denominator": "total_male_edu_pop_count", "method": "percentage"},
    "level_3_qualifications_male_perc": {"source": "level_3_qualifications_male_count", "denominator": "total_male_edu_pop_count", "method": "percentage"},
    "level_4_qualifications_or_above_male_perc": {"source": "level_4_qualifications_or_above_male_count", "denominator": "total_male_edu_pop_count", "method": "percentage"},
    "other_qualifications_male_perc": {"source": "other_qualifications_male_count", "denominator": "total_male_edu_pop_count", "method": "percentage"}
}

#Age
age_config = {
    # 1. Total Population
    "aged_0_to_9_years_perc": {"source": "aged_0_to_9_years_count", "denominator": "2021_total_population_count", "method": "percentage"},
    "aged_10_to_15_years_perc": {"source": "aged_10_to_15_years_count", "denominator": "2021_total_population_count", "method": "percentage"},
    "aged_16_to_19_years_perc": {"source": "aged_16_to_19_years_count", "denominator": "2021_total_population_count", "method": "percentage"},
    "aged_20_to_29_years_perc": {"source": "aged_20_to_29_years_count", "denominator": "2021_total_population_count", "method": "percentage"},
    "aged_30_to_39_years_perc": {"source": "aged_30_to_39_years_count", "denominator": "2021_total_population_count", "method": "percentage"},
    "aged_40_to_49_years_perc": {"source": "aged_40_to_49_years_count", "denominator": "2021_total_population_count", "method": "percentage"},
    "aged_50_to_59_years_perc": {"source": "aged_50_to_59_years_count", "denominator": "2021_total_population_count", "method": "percentage"},
    "aged_60_to_64_years_perc": {"source": "aged_60_to_64_years_count", "denominator": "2021_total_population_count", "method": "percentage"},
    "aged_65_to_69_years_perc": {"source": "aged_65_to_69_years_count", "denominator": "2021_total_population_count", "method": "percentage"},
    "aged_70_to_79_years_perc": {"source": "aged_70_to_79_years_count", "denominator": "2021_total_population_count", "method": "percentage"},
    "aged_80_and_above_years_perc": {"source": "aged_80_and_above_years_count", "denominator": "2021_total_population_count", "method": "percentage"},

    # 2. Female Population
    "aged_0_to_9_years_female_perc": {"source": "aged_0_to_9_years_female_count", "denominator": "2021_female_population_count", "method": "percentage"},
    "aged_10_to_15_years_female_perc": {"source": "aged_10_to_15_years_female_count", "denominator": "2021_female_population_count", "method": "percentage"},
    "aged_16_to_19_years_female_perc": {"source": "aged_16_to_19_years_female_count", "denominator": "2021_female_population_count", "method": "percentage"},
    "aged_20_to_29_years_female_perc": {"source": "aged_20_to_29_years_female_count", "denominator": "2021_female_population_count", "method": "percentage"},
    "aged_30_to_39_years_female_perc": {"source": "aged_30_to_39_years_female_count", "denominator": "2021_female_population_count", "method": "percentage"},
    "aged_40_to_49_years_female_perc": {"source": "aged_40_to_49_years_female_count", "denominator": "2021_female_population_count", "method": "percentage"},
    "aged_50_to_59_years_female_perc": {"source": "aged_50_to_59_years_female_count", "denominator": "2021_female_population_count", "method": "percentage"},
    "aged_60_to_64_years_female_perc": {"source": "aged_60_to_64_years_female_count", "denominator": "2021_female_population_count", "method": "percentage"},
    "aged_65_to_69_years_female_perc": {"source": "aged_65_to_69_years_female_count", "denominator": "2021_female_population_count", "method": "percentage"},
    "aged_70_to_79_years_female_perc": {"source": "aged_70_to_79_years_female_count", "denominator": "2021_female_population_count", "method": "percentage"},
    "aged_80_and_above_years_female_perc": {"source": "aged_80_and_above_years_female_count", "denominator": "2021_female_population_count", "method": "percentage"},

    # 3. Male Population
    "aged_0_to_9_years_male_perc": {"source": "aged_0_to_9_years_male_count", "denominator": "2021_male_population_count", "method": "percentage"},
    "aged_10_to_15_years_male_perc": {"source": "aged_10_to_15_years_male_count", "denominator": "2021_male_population_count", "method": "percentage"},
    "aged_16_to_19_years_male_perc": {"source": "aged_16_to_19_years_male_count", "denominator": "2021_male_population_count", "method": "percentage"},
    "aged_20_to_29_years_male_perc": {"source": "aged_20_to_29_years_male_count", "denominator": "2021_male_population_count", "method": "percentage"},
    "aged_30_to_39_years_male_perc": {"source": "aged_30_to_39_years_male_count", "denominator": "2021_male_population_count", "method": "percentage"},
    "aged_40_to_49_years_male_perc": {"source": "aged_40_to_49_years_male_count", "denominator": "2021_male_population_count", "method": "percentage"},
    "aged_50_to_59_years_male_perc": {"source": "aged_50_to_59_years_male_count", "denominator": "2021_male_population_count", "method": "percentage"},
    "aged_60_to_64_years_male_perc": {"source": "aged_60_to_64_years_male_count", "denominator": "2021_male_population_count", "method": "percentage"},
    "aged_65_to_69_years_male_perc": {"source": "aged_65_to_69_years_male_count", "denominator": "2021_male_population_count", "method": "percentage"},
    "aged_70_to_79_years_male_perc": {"source": "aged_70_to_79_years_male_count", "denominator": "2021_male_population_count", "method": "percentage"},
    "aged_80_and_above_years_male_perc": {"source": "aged_80_and_above_years_male_count", "denominator": "2021_male_population_count", "method": "percentage"}
}

In [17]:
# ==========================================
# 3. GEOGRAPHY & MATH ENGINES
# ==========================================
def generate_geographies(df, geo_cols, include_flags):
    geographies = {}
    
    if include_flags.get('england_wales'):
        geographies['England & Wales'] = df.assign(Area='England & Wales').groupby('Area')
        
    if include_flags.get('england'):
        geographies['England'] = df[df[geo_cols['region']].notna()].assign(Area='England').groupby('Area')
        
    if include_flags.get('wales'):
        geographies['Wales'] = df[df[geo_cols['region']].isna()].assign(Area='Wales').groupby('Area')
        
    regions = [
        'North East', 'North West', 'Yorkshire and The Humber', 
        'East Midlands', 'West Midlands', 'East of England', 
        'London', 'South East', 'South West'
    ]
    for region in regions:
        flag_key = region.lower().replace(' ', '_')
        if include_flags.get(flag_key):
            geographies[region] = df[df[geo_cols['region']] == region].assign(Area=region).groupby('Area')
            
    # NEW: Local Authorities Grouping
    if include_flags.get('local_authorities'):
        lad_col = geo_cols['local_authority']
        lad_df = df.assign(Area=df[lad_col])
        geographies['Local Authorities'] = lad_df.groupby([lad_col, 'Area'])
            
    # UPDATED: Wards Grouping (Simplified to just use unique codes)
    if include_flags.get('wards'):
        ward_col = geo_cols['ward']
        ward_df = df.assign(Area=df[ward_col])
        geographies['Wards'] = ward_df.groupby([ward_col, 'Area'])

    if include_flags.get('individual'):
        id_col = geo_cols['id_field']
        ind_df = df.assign(Area=df[id_col])
        geographies['Individual'] = ind_df.groupby([id_col, 'Area'])

    return geographies

def apply_calculations(grouped_df, config):
    # 1. Figure out which columns we need to sum
    columns_to_sum = set()
    for output_field, rules in config.items():
        if rules['method'] == 'percentage':
            columns_to_sum.add(rules['source'])
            columns_to_sum.add(rules['denominator'])
            
    # Sum the required columns across the geographical group
    aggregated_df = grouped_df[list(columns_to_sum)].sum().reset_index()
    
    # 2. Calculate the final fields
    final_output_df = pd.DataFrame()
    final_output_df['Area'] = aggregated_df['Area']
    
    for output_field, rules in config.items():
        if rules['method'] == 'percentage':
            num = aggregated_df[rules['source']]
            den = aggregated_df[rules['denominator']]
            
            # Calculate percentage and fill any division-by-zero errors with 0
            final_output_df[output_field] = (num / den).fillna(0) * 100
            
    return final_output_df.to_dict(orient='records')

In [18]:
def export_to_json(folder_name, final_data_list):
    # Base path specified in your original prompt
    base_path = r"C:\Users\abhimanya.achara\OneDrive - priorpartners.com\Documents\GitHub\Place_Plus\src\app\national"
    
    # NEW: Added 'data' to the folder path so it becomes ethnicity/data/
    target_dir = os.path.join(base_path, folder_name, 'data')
    
    # Create folder structure. 
    # NOTE: exist_ok=True is the magic parameter here. It ensures the code 
    # silently continues without throwing an error if the folder already exists.
    os.makedirs(target_dir, exist_ok=True)
    
    # Structure the JSON
    output_json = {
        "datasets": [
            {
                "name": f"People Census 2021 {folder_name.capitalize()} Benchmark",
                "id": "benchmarks",
                "title": f"People Census 2021 {folder_name.capitalize()} Benchmark",
                "description": f"Benchmark data for {folder_name}",
                "data": final_data_list
            }
        ]
    }
    
    # Define the file path (e.g., benchmark-ethnicity.json)
    file_path = os.path.join(target_dir, f"benchmark-{folder_name}.json")
    
    # Write to JSON. 
    # NOTE: The 'w' (write) mode automatically overwrites the file if it already exists. 
    # If you ever wanted to append, you would use 'a', but 'w' does exactly what you need.
    with open(file_path, 'w') as f:
        json.dump(output_json, f, indent=2)
    
    print(f"Successfully created/updated: {file_path}")

In [28]:
# ==========================================
# 5. MAIN EXECUTION PIPELINE
# ==========================================
def run_pipeline():
    schema = 'uk_new'
    layer = 'people_ons_census2021_lsoa2021_age_alternate_bands'
    folder_name = 'age'
    
    # NEW: Define your spatial columns here so they are easy to change
    geo_columns = {
        'id_field': 'lsoa21cd',
        'region': 'rgn22nm',
        'ward': 'wd22cd',             
        'local_authority': 'lad22nm'  
    }
    
    # Toggles for all 13 geographical splits
    include_flags = {
        'england_wales': True,
        'england': True,
        'wales': True,
        'north_east': True,
        'north_west': True,
        'yorkshire_and_the_humber': True,
        'east_midlands': True,
        'west_midlands': True,
        'east_of_england': True,
        'london': True,
        'south_east': True,
        'south_west': True,
        'local_authorities': True,
        'wards': True,     
        'individual': True  
    }
    
    print("1. Fetching data from PostGIS...")
    
    # Dynamically figure out exactly which columns we need
    needed_columns = set()
    
    # Add the spatial columns (lsoa21cd, rgn22nm, etc.)
    for key, col_name in geo_columns.items():
        needed_columns.add(col_name)
        
    # Add the data columns (asian_count, total_age_pop, etc.)
    for output_field, rules in age_config.items():
        needed_columns.add(rules['source'])
        needed_columns.add(rules['denominator'])
        
    # Pass the list of needed columns to the fetch function
    df = fetch_data(schema, layer, list(needed_columns))
    
    print("2. Generating geographical splits...")
    # NEW: Pass the geo_columns dictionary instead of just the id_field
    geographies = generate_geographies(df, geo_columns, include_flags)
    
    # ... rest of the export code ...
    
    final_json_data = []
    
    print("3. Calculating metrics...")
    for geo_name, grouped_data in geographies.items():
        calculated_records = apply_calculations(grouped_data, age_config)
        final_json_data.extend(calculated_records)
        
    print("4. Exporting to JSON...")
    export_to_json(folder_name, final_json_data)
    print("Pipeline complete!")

# RUN THE SCRIPT
if __name__ == "__main__":
    run_pipeline()

1. Fetching data from PostGIS...
Executing query: SELECT [Needed Columns] FROM uk_new.people_ons_census2021_lsoa2021_age_alternate_bands
2. Generating geographical splits...
3. Calculating metrics...
4. Exporting to JSON...
Successfully created/updated: C:\Users\abhimanya.achara\OneDrive - priorpartners.com\Documents\GitHub\Place_Plus\src\app\national\age\data\benchmark-age.json
Pipeline complete!
